In [1]:
%pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Inicialização do PySpark
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

conf = SparkConf()
conf.set('spark.jars.packages', 'org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.11.901')
conf.set('spark.hadoop.fs.s3a.aws.credentials.provider', 'com.amazonaws.auth.InstanceProfileCredentialsProvider')

spark = SparkSession.builder.config(conf=conf).getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.7/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ceb1837f-0111-4601-84cc-1cfddbc22ac1;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 416ms :: artifacts dl 17ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.901 by [com.amazonaws#aws-java-sdk-bundle;1.12.262] in [default]
	---------------------------------------------------------------------
	|     

In [3]:
tb_orders = spark.read.csv("s3a://last-mile-optimization-trusted/dataset-orders/final_table_.csv/", header=True, inferSchema=True)

26/04/13 01:47:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [4]:
from pyspark.sql import functions as F

# 1. Cálculos de Diferença de Tempo (Transformando timestamps em métricas úteis)
df_final = tb_orders.withColumn(
    "approval_time_hours", 
    (F.unix_timestamp("order_approved_at") - F.unix_timestamp("order_purchase_timestamp")) / 3600
).withColumn(
    "handling_time_hours",
    (F.unix_timestamp("order_delivered_carrier_date") - F.unix_timestamp("order_approved_at")) / 3600
).withColumn(
    "shipping_delay_hours",
    (F.unix_timestamp("order_delivered_carrier_date") - F.unix_timestamp("shipping_limit_date")) / 3600
).withColumn(
    "delivery_time_days",
    (F.unix_timestamp("order_delivered_customer_date") - F.unix_timestamp("order_purchase_timestamp")) / 86400
)

# 2. Extração de Componentes de Data/Hora e Intervalos Requisitados
df_final = df_final.withColumn("purchase_hour", F.hour("order_purchase_timestamp")) \
                   .withColumn("purchase_day_of_week", F.dayofweek("order_purchase_timestamp") - 1) \
                   .withColumn("is_weekend", F.when(F.col("purchase_day_of_week").isin(5, 6), 1).otherwise(0))

# Adicionando os intervalos de 6h (Madrugada, Manhã, Tarde, Noite)
df_final = df_final.withColumn(
    "interval_code_delivered_carrier",
    F.when((F.hour("order_delivered_carrier_date") >= 0) & (F.hour("order_delivered_carrier_date") < 6), 1)
     .when((F.hour("order_delivered_carrier_date") >= 6) & (F.hour("order_delivered_carrier_date") < 12), 2)
     .when((F.hour("order_delivered_carrier_date") >= 12) & (F.hour("order_delivered_carrier_date") < 18), 3)
     .when(F.hour("order_delivered_carrier_date").isNotNull(), 4) # Senão for nulo, cai na Noite
).withColumn(
    "interval_code_delivered_customer",
    F.when((F.hour("order_delivered_customer_date") >= 0) & (F.hour("order_delivered_customer_date") < 6), 1)
     .when((F.hour("order_delivered_customer_date") >= 6) & (F.hour("order_delivered_customer_date") < 12), 2)
     .when((F.hour("order_delivered_customer_date") >= 12) & (F.hour("order_delivered_customer_date") < 18), 3)
     .when(F.hour("order_delivered_customer_date").isNotNull(), 4)
)

# 3. Cálculos de Geolocalização e Logística
df_final = df_final.withColumn(
    "distance_km",
    F.expr("""
        6371 * acos(
            cos(radians(customer_geolocation_lat)) * cos(radians(seller_geolocation_lat)) * cos(radians(seller_geolocation_lng) - radians(customer_geolocation_lng)) + 
            sin(radians(customer_geolocation_lat)) * sin(radians(seller_geolocation_lat))
        )
    """)
).withColumn(
    "same_city", 
    F.when(F.col("customer_city") == F.col("seller_city"), 1).otherwise(0)
)

# 4. Renomeando categoria
df_final = df_final.withColumnRenamed("category_name", "category_name_encoded")

# 5. ARREDONDAMENTO (3 casas decimais)
colunas_para_arredondar = [
    "approval_time_hours", "handling_time_hours", "shipping_delay_hours", 
    "delivery_time_days", "price", "freight_value", 
    "product_weight_g", "volume_cm3", "distance_km"
]

for col_name in colunas_para_arredondar:
    df_final = df_final.withColumn(col_name, F.round(F.col(col_name), 3))

# 6. Seleção Final Restrita (Incluindo as novas colunas)
colunas_finais = [
    "order_delivered_carrier_date", 
    "interval_code_delivered_carrier", 
    "order_delivered_customer_date", 
    "interval_code_delivered_customer",
    "approval_time_hours",
    "handling_time_hours", 
    "shipping_delay_hours", 
    "delivery_time_days",
    "purchase_hour", 
    "purchase_day_of_week", 
    "is_weekend", 
    "price",
    "freight_value", 
    "product_weight_g", 
    "volume_cm3", 
    "distance_km",
    "same_city", 
    "review_score", 
    "delivered_on_time",
    "customer_geolocation_lat", 
    "customer_geolocation_lng", 
    "seller_geolocation_lat", 
    "seller_geolocation_lng"
]

# Aplicando o filtro final
df_export = df_final.select(*colunas_finais)

# Preview e conferência
df_export.show(5)
print(f"Total de colunas: {len(df_export.columns)}")

+----------------------------+-------------------------------+-----------------------------+--------------------------------+-------------------+-------------------+--------------------+------------------+-------------+--------------------+----------+------+-------------+----------------+----------+-----------+---------+------------+-----------------+------------------------+------------------------+----------------------+----------------------+
|order_delivered_carrier_date|interval_code_delivered_carrier|order_delivered_customer_date|interval_code_delivered_customer|approval_time_hours|handling_time_hours|shipping_delay_hours|delivery_time_days|purchase_hour|purchase_day_of_week|is_weekend| price|freight_value|product_weight_g|volume_cm3|distance_km|same_city|review_score|delivered_on_time|customer_geolocation_lat|customer_geolocation_lng|seller_geolocation_lat|seller_geolocation_lng|
+----------------------------+-------------------------------+-----------------------------+--------

In [5]:
df_export.coalesce(1) \
    .write \
    .option('header', 'true') \
    .mode('overwrite') \
    .csv('s3a://last-mile-optimization-trusted/dataset-orders/final_table_orders_numeric.csv')

spark.stop()

26/04/13 01:47:40 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/13 01:47:41 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
